<a href="https://colab.research.google.com/github/alakaramkesh/Character-Level-GPT/blob/main/LLM_Poem_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
! pip install torch

# ***Building the Model:***

In [25]:
import json
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import time

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [27]:
!ls /content
# if you do not see these three files: train_ids.npy  val_ids.npy	vocab.json , please upload them

checkpoints  drive  sample_data  train_ids.npy	val_ids.npy  vocab.json


Read the files created in the last step. (the .npy ones and the vocab .json file)

In [28]:
base = Path("/content")

with open(base / "vocab.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)
# loading the numpy arrays of corpus
train_ids = np.load(base / "train_ids.npy")
val_ids = np.load(base / "val_ids.npy")
# turn np arrays to tensor
train_ids = torch.tensor(train_ids, dtype=torch.long)
val_ids = torch.tensor(val_ids, dtype=torch.long)
print("vocab size:", len(vocab["stoi"]))
print("train shape:", train_ids.shape)
print("val shape:", val_ids.shape)

vocab size: 46
train shape: torch.Size([607786])
val shape: torch.Size([70512])


Here the method creates batches of input and output.
(Shifting the gold output by one character)

In [30]:
def get_batch(split, train_ids, val_ids, batch_size, context_len, device):
    data = train_ids if split == "train" else val_ids
    ix = torch.randint(0, len(data) - context_len - 1, (batch_size,))
    x = torch.stack([data[i:i+context_len] for i in ix])
    y = torch.stack([data[i+1:i+context_len+1] for i in ix]) # shifting x by one position
    return x.to(device), y.to(device)

Causal self-attention layer with two options: manual masked attention or PyTorch SDPA.

In [31]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, attention_mode="sdpa"):
        super().__init__()
        #checks
        assert d_model % n_heads == 0
        assert attention_mode in ["sdpa", "manual"]

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.attention_mode = attention_mode

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def _shape(self, x, B, T):
        # (B, T, C) -> (B, H, T, D)
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    # traditional approach based on masked attention with a triangular causal mask
    def _manual_attention(self, q, k, v, x):
        B, H, T, D = q.shape
        # raw attention scores
        scores = q @ k.transpose(-2, -1)   # (B, H, T, T)
        # scale
        scores = scores / math.sqrt(D)
        # causal mask
        mask = torch.tril(
            torch.ones(T, T, device=x.device, dtype=torch.bool)
        )
        scores = scores.masked_fill(~mask, float("-inf"))
        # normalize
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights) # prevent overfitting
        # weighted sum of values
        attn_out = attn_weights @ v   # (B, H, T, D)
        return attn_out

    # torch.nn.functional.scaled_dot_product_attention
    def _sdpa_attention(self, q, k, v):
        return F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.attn_dropout.p if self.training else 0.0,
            is_causal=True
        )

    def forward(self, x):
        B, T, C = x.shape
        # build Q, K, V
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # split into heads
        q = self._shape(q, B, T)
        k = self._shape(k, B, T)
        v = self._shape(v, B, T)
        # choose attention implementation
        if self.attention_mode == "manual":
            attn_out = self._manual_attention(q, k, v, x)
        else:
            attn_out = self._sdpa_attention(q, k, v)
        # merge heads: (B, H, T, D) -> (B, T, C)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, C)
        # final output projection
        out = self.out_proj(attn_out)
        out = self.resid_dropout(out)
        return out

Here is the class for the feed forward layer of the architecture.

In [32]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()

        if d_ff is None:
            d_ff = 4 * d_model

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

A single Transformer block with Pre-LN, causal self-attention, and a feed-forward network.

In [33]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1, attention_mode="sdpa"):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout,
            attention_mode=attention_mode
        )

        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

    def forward(self, x):
        # Pre-LN attention + residual
        x = x + self.attn(self.ln1(x))
        # Pre-LN FFN + residual
        x = x + self.ff(self.ln2(x))
        return x

# ***Training the Model***
The main class is the CharGPT class.

In [34]:
class CharGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        context_len,
        d_model,
        n_heads,
        n_layers,
        d_ff=None,
        dropout=0.1,
        attention_mode="sdpa"
    ):
        super().__init__()
        self.context_len = context_len
        self.d_model = d_model
        # token embedding: maps each character id to a vector
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # learned positional embedding
        self.pos_emb = nn.Embedding(context_len, d_model)
        # stack of transformer blocks
        self.blocks = nn.Sequential(*[
            TransformerBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout,
                attention_mode=attention_mode
            )
            for _ in range(n_layers)
        ])
        # final layer norm
        self.ln_f = nn.LayerNorm(d_model)
        # output projection to vocabulary size
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, idx, targets=None):
        # idx shape: (B, T)
        B, T = idx.shape
        assert T <= self.context_len, "Sequence length is bigger than block size"
        # positions: [0, 1, 2, ..., T-1]
        pos = torch.arange(0, T, device=idx.device)
        # token embeddings: (B, T, d_model)
        tok = self.token_emb(idx)
        # positional embeddings: (T, d_model)
        pos = self.pos_emb(pos)
        # broadcast add -> (B, T, d_model)
        x = tok + pos
        # pass through transformer blocks
        x = self.blocks(x)
        # final layer norm
        x = self.ln_f(x)
        # logits over vocabulary: (B, T, vocab_size)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, -1),
                targets.view(B * T)
            )
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            # Keep only the last context_len tokens
            idx_cond = idx[:, -self.context_len:]
            # Forward pass
            logits, _ = self(idx_cond)
            # Take logits of the last time step
            logits = logits[:, -1, :]
            # Apply temperature
            logits = logits / temperature
            # Optional top-k filtering
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            # Convert to probabilities
            probs = torch.softmax(logits, dim=-1)
            # Sample next token
            next_token = torch.multinomial(probs, num_samples=1)
            # Append to sequence
            idx = torch.cat((idx, next_token), dim=1)
        self.train()
        return idx

` BASE HYPERPARAMETERS:`

In [ ]:
base_config = {
    "d_model": 256,
    "n_heads": 4,
    "n_layers": 6,
    "dropout": 0.1,
    "batch_size": 32,
    "context_len": 256,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,
    "max_steps": 10000,
    "log_interval": 100,
    "eval_interval": 200,
    "eval_iters": 10,
    "sample_interval": 1000,
    "checkpoint_interval": 1000,
    "warmup_steps": 500,
    "grad_clip": 1.0,
    "attention_mode": "sdpa",
}

In [35]:
# Training setup
max_steps = 10000
log_interval = 100
eval_interval = 200
eval_iters = 10
sample_interval = 1000
checkpoint_interval = 1000

# Optimizer setup
learning_rate = 3e-4
weight_decay = 0.1
grad_clip = 1.0

# Warm-up
warmup_steps = 500

# Checkpoint / tracking
save_dir = "checkpoints"
os.makedirs(save_dir, exist_ok=True)

best_val_loss = float("inf")
history = []

In [36]:
vocab_size = len(vocab["stoi"])
d_model = 256 # vector dimension
n_heads = 4 # Number of attention heads
n_layers = 6
dropout = 0.1
batch_size = 32
context_len = 256 # context length

model = CharGPT(
    vocab_size=vocab_size,
    context_len=context_len,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=4 * d_model,
    dropout=dropout,
    attention_mode="sdpa"
).to(device)
# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

In [37]:
itos = vocab["itos"]
stoi = vocab["stoi"]

In [ ]:
def estimate_loss(model, train_ids, val_ids, batch_size, context_len, device, eval_iters=20):
    out = {}
    model.eval() # switch to evaluation mode
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split, train_ids, val_ids, batch_size, context_len, device)
            with torch.no_grad(): # disable gradient tracking here
                logits, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train() # back to training mode
    return out

In [39]:
def decode(ids):
    # Convert token ids back to text.
    return "".join(itos[str(i)] for i in ids)

def encode(text):
    # Convert text to token ids.
    unk_id = vocab["stoi"].get("<unk>", None)
    ids = []
    for ch in text:
        if ch in stoi:
            ids.append(stoi[ch])
        elif unk_id is not None:
            ids.append(unk_id)
    return ids

def generate_text(prompt="\n", max_new_tokens=300, temperature=1.0, top_k=20):
    # Generate text from a prompt.
    model.eval()
    start_ids = encode(prompt)
    if len(start_ids) == 0:
        start_ids = [stoi["\n"]] if "\n" in stoi else [0]
    x = torch.tensor(start_ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        y = model.generate(
            x,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k
        )
    text = decode(y[0].tolist())
    model.train()
    return text

def loss_to_bpc(loss):
    return loss / math.log(2)

def get_lr(step):
    #Linear warm-up for the first warmup_steps.
    if step < warmup_steps:
        return learning_rate * (step + 1) / warmup_steps
    return learning_rate

def save_checkpoint(path, step, val_loss):
    torch.save("config": {
    "block_size": context_len,
    "d_model": d_model,
    "n_heads": n_heads,
    "n_layers": n_layers,
    "dropout": dropout,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "attention_mode": "sdpa",
}, path)

NOW TRAIN:

and also monitor what is happening

In [40]:
model.train()
start_time = time.time()
running_loss = 0.0

for step in range(max_steps):
    # Update learning rate with warm-up
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr
    # Get one batch from training data
    xb, yb = get_batch("train")
    # Forward pass
    logits, loss = model(xb, yb)
    # Clear old gradients
    optimizer.zero_grad(set_to_none=True)
    # Backward pass
    loss.backward()
    # Gradient clipping for stability
    #torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    # Update parameters
    optimizer.step()
    running_loss += loss.item()
    # Log average training loss
    if (step + 1) % log_interval == 0:
        avg_train_loss = running_loss / log_interval
        elapsed = time.time() - start_time
        print(
            f"step {step+1}/{max_steps} | "
            f"avg train loss = {avg_train_loss:.4f} | "
            f"lr = {lr:.6f} | "
            f"time = {elapsed:.1f}s"
        )
        running_loss = 0.0

    # Evaluation
    if (step + 1) % eval_interval == 0:
        losses = estimate_loss(eval_iters=eval_iters)
        train_bpc = loss_to_bpc(losses["train"])
        val_bpc = loss_to_bpc(losses["val"])

        print(
            f"[eval] step {step+1} | "
            f"train loss = {losses['train']:.4f} | "
            f"val loss = {losses['val']:.4f} | "
            f"train bpc = {train_bpc:.4f} | "
            f"val bpc = {val_bpc:.4f}"
        )

        # Save history for later plotting/report
        history.append({
            "step": step + 1,
            "train_loss": losses["train"],
            "val_loss": losses["val"],
            "train_bpc": train_bpc,
            "val_bpc": val_bpc,
            "lr": lr,
        })

        # Save best checkpoint
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            save_checkpoint(
                os.path.join(save_dir, "best_model.pt"),
                step + 1,
                losses["val"]
            )
            print(f"New best model saved at step {step+1}")

    # Sample generation every 1k steps
    if (step + 1) % sample_interval == 0:
        sample_text = generate_text(
            prompt="\n",
            max_new_tokens=400,
            temperature=1.0,
            top_k=20
        )
        print("\n--- SAMPLE ---\n")
        print(sample_text)
        print("\n--------------\n")

    # Regular checkpoint every 1k steps
    if (step + 1) % checkpoint_interval == 0:
        ckpt_path = os.path.join(save_dir, f"checkpoint_step_{step+1}.pt")
        val_for_ckpt = history[-1]["val_loss"] if len(history) > 0 else None
        save_checkpoint(ckpt_path, step + 1, val_for_ckpt)
        print(f"Checkpoint saved to {ckpt_path}")

step 100/10000 | avg train loss = 3.2484 | lr = 0.000060 | time = 10.3s
step 200/10000 | avg train loss = 2.6321 | lr = 0.000120 | time = 20.8s
[eval] step 200 | train loss = 2.5778 | val loss = 2.5827 | train bpc = 3.7190 | val bpc = 3.7261
New best model saved at step 200
step 300/10000 | avg train loss = 2.5519 | lr = 0.000180 | time = 32.2s
step 400/10000 | avg train loss = 2.5198 | lr = 0.000240 | time = 43.2s
[eval] step 400 | train loss = 2.5112 | val loss = 2.5140 | train bpc = 3.6229 | val bpc = 3.6269
New best model saved at step 400
step 500/10000 | avg train loss = 2.4975 | lr = 0.000300 | time = 55.2s
step 600/10000 | avg train loss = 2.4605 | lr = 0.000300 | time = 66.2s
[eval] step 600 | train loss = 2.4286 | val loss = 2.4140 | train bpc = 3.5037 | val bpc = 3.4827
New best model saved at step 600
step 700/10000 | avg train loss = 2.3963 | lr = 0.000300 | time = 77.7s


KeyboardInterrupt: 

In [ ]:
with open("training_history.json", "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print("Training history saved.")

The plots:

In [ ]:
import matplotlib.pyplot as plt

with open("training_history.json", "r", encoding="utf-8") as f:
    history = json.load(f)

steps = [x["step"] for x in history]
train_loss = [x["train_loss"] for x in history]
val_loss = [x["val_loss"] for x in history]
train_bpc = [x["train_bpc"] for x in history]
val_bpc = [x["val_bpc"] for x in history]

# Loss curve
plt.figure(figsize=(8, 5))
plt.plot(steps, train_loss, label="Train loss")
plt.plot(steps, val_loss, label="Validation loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

# BPC curve
plt.figure(figsize=(8, 5))
plt.plot(steps, train_bpc, label="Train BPC")
plt.plot(steps, val_bpc, label="Validation BPC")
plt.xlabel("Training step")
plt.ylabel("Bits per character")
plt.title("Training and Validation BPC")
plt.legend()
plt.show()